In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage
from langgraph.graph.message import add_messages  ## Optimized reducer with BaseMessage Class
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from my_llm import *
from langchain_core.runnables import RunnableConfig
import sqlite3

In [2]:
class TranslateState(TypedDict):
    input_text: str
    generated_english_text: str
    Hindi_text: str

class TranslateEnglishState(TypedDict):
    input_text: str
    Hindi_text: str

In [ ]:
## Creating functions for Nodes

def generate_text(state: TranslateState) -> str:
    # This is where you would call your LLM to generate English text from the input text
    # For demonstration, we'll just return a dummy string
    return {"generated_english_text": f"Generated English text for: {state['input_text']}"}


def translate_english_text(state: TranslateEnglishState) -> str:
    # This is where you would call your LLM to generate English text from the input text
    # For demonstration, we'll just return a dummy string
    return {"Hindi_text": f"Generated Hindi text for: {state['input_text']}"}



In [15]:
## Define SubGraph

subgraph = StateGraph(TranslateEnglishState)
subgraph.add_node("translate_english_text", translate_english_text)

subgraph.set_entry_point("translate_english_text")
subgraph.set_finish_point("translate_english_text")

child_workflow = subgraph.compile()




In [16]:
child_workflow.invoke({"input_text": "What is the capital of France?"})

{'input_text': 'What is the capital of France?',
 'Hindi_text': 'Generated Hindi text for: What is the capital of France?'}

In [20]:
def translate_answer(state: TranslateState) -> str:

    result = child_workflow.invoke({"input_text": state["generated_english_text"]})
    return {"Hindi_text": result["Hindi_text"]}

In [ ]:
## define main graph

graph = StateGraph(TranslateState)
graph.add_node("generate_text", generate_text)
graph.add_node("translate_answer",translate_answer)


graph.set_entry_point("generate_text")
graph.add_edge("generate_text", "translate_answer")
graph.set_finish_point("translate_answer")

parent_graph = graph.compile()




In [22]:
parent_graph.invoke({"input_text": "What is the capital of France?"})

{'input_text': 'What is the capital of France?',
 'generated_english_text': 'Generated English text for: What is the capital of France?',
 'Hindi_text': 'Generated Hindi text for: Generated English text for: What is the capital of France?'}